<a href="https://colab.research.google.com/github/ghadaPoly/Projet_ML/blob/main/3_Feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⛳ Phase 3 : FEATURE ENGINEERING

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

In [ ]:
df = pd.read_csv("diabetic_data_cleaned.csv")

print("Dimensions du dataset :", df.shape)

Dimensions du dataset : (101763, 44)


In [ ]:
print(df.dtypes)

patient_nbr                  int64
race                        object
gender                      object
age                         object
admission_type_id            int64
discharge_disposition_id     int64
admission_source_id          int64
time_in_hospital             int64
num_lab_procedures           int64
num_procedures               int64
num_medications              int64
number_outpatient            int64
number_emergency             int64
number_inpatient             int64
diag_1                      object
diag_2                      object
diag_3                      object
number_diagnoses             int64
max_glu_serum               object
A1Cresult                   object
metformin                   object
repaglinide                 object
nateglinide                 object
chlorpropamide              object
glimepiride                 object
acetohexamide               object
glipizide                   object
glyburide                   object
tolbutamide         

##  Variables liées aux médicaments

Dans le dataset, plusieurs colonnes décrivent l’état de différents médicaments (No, Steady, Up, Down).  
Cependant, ces colonnes sont nombreuses et peu exploitables individuellement.

Afin de simplifier et d’extraire une information pertinente, nous avons créé deux nouvelles variables :

- **`nb_meds_actifs`** : représente le *nombre de médicaments actifs* pour chaque patient (tous les médicaments différents de "No").
- **`nb_meds_modifies`** : représente le *nombre de médicaments dont la dose a été modifiée* ("Up" ou "Down").

*Cette transformation permet de réduire la dimension des données tout en conservant l’information essentielle sur la complexité du traitement.*

Enfin, les colonnes originales ont été supprimées afin d’éviter la redondance et le bruit dans le modèle.

In [ ]:
colonnes_meds = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
                 'glimepiride', 'glipizide', 'glyburide', 'pioglitazone',
                 'rosiglitazone', 'acarbose', 'insulin', 'glyburide-metformin']

df['nb_meds_actifs']   = (df[colonnes_meds] != 'No').sum(axis=1)
df['nb_meds_modifies'] = df[colonnes_meds].isin(['Up', 'Down']).sum(axis=1)

In [ ]:
df = df.drop(columns=colonnes_meds)

In [ ]:

# ================================================================
#  SUPPRESSION DES COLONNES INUTILES (APRÈS FEATURE ENGINEERING)
# ================================================================

colonnes_a_supprimer_final = [

    # médicaments redondants (déjà résumés)
    'acetohexamide', 'tolbutamide', 'miglitol', 'troglitazone',
    'tolazamide', 'glipizide-metformin', 'glimepiride-pioglitazone',
    'metformin-rosiglitazone', 'metformin-pioglitazone'
]
print("Nb colonnes avant suppression :", df.shape[1])
# Sécurité (éviter erreurs)
colonnes_a_supprimer_final = [c for c in colonnes_a_supprimer_final if c in df.columns]

df = df.drop(columns=colonnes_a_supprimer_final)

print("Colonnes supprimées :", colonnes_a_supprimer_final)
print("Nb colonnes restantes :", df.shape[1])

Nb colonnes avant suppression : 34
Colonnes supprimées : ['acetohexamide', 'tolbutamide', 'miglitol', 'troglitazone', 'tolazamide', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']
Nb colonnes restantes : 25


Transformation de la variable âge

La variable **âge** est initialement représentée sous forme d’intervalles (ex : [60-70)), ce qui la rend difficilement exploitable directement par les modèles de Machine Learning.

Nous avons donc transformé ces intervalles en valeurs numériques en utilisant leur **point médian** :

- Exemple : *[60-70) → 65*

Cette approche permet de conserver une approximation réaliste de l’âge tout en rendant la variable exploitable.

De plus, nous avons créé une nouvelle variable :

- **`is_senior`** : indicateur binaire prenant la valeur :
  - *1* si le patient est âgé de 65 ans ou plus  
  - *0* sinon

*Cette variable permet de capturer le risque accru associé aux patients âgés, souvent plus susceptibles de complications ou de ré-hospitalisation.*

La colonne originale **`age`** a ensuite été supprimée pour éviter toute redondance.

“Les intervalles d’âge ont été transformés en valeurs numériques en utilisant leur point médian, afin de rendre la variable exploitable. Un indicateur binaire ‘senior’ a également été ajouté pour capturer le risque accru lié à l’âge.”

In [ ]:
age_map = {'[0-10)':5,'[10-20)':15,'[20-30)':25,'[30-40)':35,'[40-50)':45,
           '[50-60)':55,'[60-70)':65,'[70-80)':75,'[80-90)':85,'[90-100)':95}

df['age_num']  = df['age'].map(age_map)
df['is_senior'] = (df['age_num'] >= 65).astype(int)

In [ ]:
df = df.drop(columns=['age'])

## Intensité et complexité du séjour hospitalier

Les variables initiales décrivent séparément différents aspects du parcours patient (nombre de visites, procédures, médicaments, etc.).  
Cependant, ces informations peuvent être enrichies en créant des variables dérivées.

Nous avons introduit les variables suivantes :

- **`total_visits`** : somme des visites (consultations externes, urgences et hospitalisations)  
  ➝ *représente le niveau global d’activité médicale du patient*

- **`procedures_per_day`** : nombre de procédures rapporté à la durée du séjour  
  ➝ *mesure l’intensité des soins reçus*

- **`meds_per_diag`** : ratio entre le nombre de médicaments et le nombre de diagnostics  
  ➝ *indique la complexité du traitement par rapport aux pathologies*

*Ces nouvelles variables permettent de capturer des relations plus fines que les variables brutes, notamment en termes d’intensité et de complexité médicale.*

Une constante (+1) a été ajoutée dans les dénominateurs afin d’éviter toute division par zéro.

Des variables dérivées ont été créées afin de capturer l’intensité et la complexité du séjour hospitalier, permettant d’enrichir l’information contenue dans les variables initiales.”

In [ ]:
df['total_visits']       = df['number_outpatient'] + df['number_emergency'] + df['number_inpatient']
df['procedures_per_day'] = df['num_procedures'] / (df['time_in_hospital'] + 1)
df['meds_per_diag']      = df['num_medications'] / (df['number_diagnoses'] + 1)
# ================================================================
#  FEATURE ENGINEERING : HISTORIQUE PATIENT
# ================================================================

# Nombre total de visites par patient
df['nb_visites_patient'] = df.groupby('patient_nbr')['patient_nbr'].transform('count')

print("Distribution de nb_visites_patient :")
print(df['nb_visites_patient'].describe())

Distribution de nb_visites_patient :
count    101763.000000
mean          2.259063
std           2.442136
min           1.000000
25%           1.000000
50%           1.000000
75%           3.000000
max          40.000000
Name: nb_visites_patient, dtype: float64


In [ ]:
df = df.drop(columns=['patient_nbr']) # data leakage

In [ ]:
df.head(20)

,race,gender,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,...,readmitted_bin,change_bin,nb_meds_actifs,nb_meds_modifies,age_num,is_senior,total_visits,procedures_per_day,meds_per_diag,nb_visites_patient
0,Caucasian,Female,6,25,1,1,41,0,1,0,...,0,0,0,0,5,0,0,0.000000,0.500000,1
1,Caucasian,Female,1,1,7,3,59,0,18,0,...,0,1,1,1,15,0,0,0.000000,1.800000,1
2,AfricanAmerican,Female,1,1,7,2,11,5,13,2,...,0,0,1,0,25,0,3,1.666667,1.857143,1
3,Caucasian,Male,1,1,7,2,44,1,16,0,...,0,1,1,1,35,0,0,0.333333,2.000000,1
4,Caucasian,Male,1,1,7,1,51,0,8,0,...,0,1,2,0,45,0,0,0.000000,1.333333,1
5,Caucasian,Male,2,1,2,3,31,6,16,0,...,0,0,1,0,55,0,0,1.500000,1.600000,1
6,Caucasian,Male,3,1,2,4,70,1,21,0,...,0,1,3,0,65,1,0,0.200000,2.625000,1
7,Caucasian,Male,1,1,7,5,73,0,12,0,...,0,0,1,0,75,1,0,0.000000,1.333333,1
8,Caucasian,Female,2,1,4,13,68,2,28,0,...,0,1,2,0,85,1,0,0.142857,3.111111,1
9,Caucasian,Female,3,3,4,12,33,3,18,0,...,0,1,2,0,95,1,0,0.230769,2.000000,1


In [ ]:
df.dtypes

,0
race,object
gender,object
admission_type_id,int64
discharge_disposition_id,int64
admission_source_id,int64
time_in_hospital,int64
num_lab_procedures,int64
num_procedures,int64
num_medications,int64
number_outpatient,int64


In [ ]:
df.columns

Index(['race', 'gender', 'admission_type_id', 'discharge_disposition_id',
       'admission_source_id', 'time_in_hospital', 'num_lab_procedures',
       'num_procedures', 'num_medications', 'number_outpatient',
       'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3',
       'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'diabetesMed',
       'readmitted_bin', 'change_bin', 'nb_meds_actifs', 'nb_meds_modifies',
       'age_num', 'is_senior', 'total_visits', 'procedures_per_day',
       'meds_per_diag', 'nb_visites_patient'],
      dtype='object')

In [ ]:
df.shape

(101763, 29)

In [ ]:
print(df.dtypes)


race                         object
gender                       object
admission_type_id             int64
discharge_disposition_id      int64
admission_source_id           int64
time_in_hospital              int64
num_lab_procedures            int64
num_procedures                int64
num_medications               int64
number_outpatient             int64
number_emergency              int64
number_inpatient              int64
diag_1                       object
diag_2                       object
diag_3                       object
number_diagnoses              int64
max_glu_serum                object
A1Cresult                    object
diabetesMed                  object
readmitted_bin                int64
change_bin                    int64
nb_meds_actifs                int64
nb_meds_modifies              int64
age_num                       int64
is_senior                     int64
total_visits                  int64
procedures_per_day          float64
meds_per_diag               

In [ ]:
# Sauvegarder le dataset nettoyé
df.to_csv('diabetic_data_after_FE.csv', index=False)

# Télécharger depuis Google Colab
from google.colab import files
files.download('diabetic_data_after_FE.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Résultat très clair !Variablevs change_binvs readmitted_binDécisionnb_meds_actifs0.71 — leakage direct0.004 — aucun lien❌ Supprimernb_meds_modifiesleakage direct (déjà confirmé)—❌ SupprimerdiabetesMedleakage direct (déjà confirmé)—❌ Supprimernb_meds_actifs = 0.71 avec change_bin, c'est du leakage pur. La moyenne passe de 0.57 → 1.88 entre les deux classes, le modèle "voit" directement la réponse.

In [ ]:
import pandas as pd
from scipy.stats import pointbiserialr

df = pd.read_csv("diabetic_data_after_FE.csv")

# Corrélation avec les deux cibles
for cible in ['change_bin', 'readmitted_bin']:
    corr, pval = pointbiserialr(df['nb_meds_actifs'], df[cible])
    print(f"nb_meds_actifs vs {cible} → corrélation={corr:.4f} | p-value={pval:.5f}")

# Distribution par classe
print("\n--- Moyenne nb_meds_actifs par classe ---")
print(df.groupby('change_bin')['nb_meds_actifs'].mean())
print(df.groupby('readmitted_bin')['nb_meds_actifs'].mean())

nb_meds_actifs vs change_bin → corrélation=0.7119 | p-value=0.00000
nb_meds_actifs vs readmitted_bin → corrélation=0.0036 | p-value=0.25735

--- Moyenne nb_meds_actifs par classe ---
change_bin
0    0.571867
1    1.885107
Name: nb_meds_actifs, dtype: float64
readmitted_bin
0    1.177355
1    1.187726
Name: nb_meds_actifs, dtype: float64


##  Conclusion du Feature Engineering

Les étapes de Feature Engineering ont permis de transformer des données brutes en variables plus informatives et plus adaptées à l’apprentissage automatique.

✔ Réduction de la dimension des données  
✔ Amélioration de l’interprétabilité  
✔ Capture de relations complexes entre variables  

 *Ces transformations constituent une étape essentielle avant le preprocessing et l’entraînement des modèles.*